In [1]:
# 单元格 1：导入依赖与配置
import json
import sys
import re
from pathlib import Path

sys.path.append(str(Path.cwd()))

from config import OLLAMA_CHAT_MODEL, SIMILARITY_THRESHOLD
from utils import ollama_chat
from prompts import (
    REASONING_PROMPT, SECOND_REASONING_PROMPT, RETHINK_PROMPT, fill_prompt
)

print(f"当前模型：{OLLAMA_CHAT_MODEL}")

当前模型：qwen3.5:9b


In [2]:
# 单元格 2：加载思维导图和知识池
mind_map_file = "data/mind_map_sample.json"
if Path(mind_map_file).exists():
    with open(mind_map_file, "r", encoding="utf-8") as f:
        mind_map_tree = json.load(f)
    print("已加载思维导图。")
else:
    # 手动构造测试树
    mind_map_tree = [
        {

        }
    ]
    print("未找到思维导图文件，使用默认测试树。")

knowledge_file = "data/retrieved_triples_combined.json"
if Path(knowledge_file).exists():
    with open(knowledge_file, "r", encoding="utf-8") as f:
        knowledge_triples = json.load(f)
    print(f"已加载知识池，共 {len(knowledge_triples)} 条三元组。")
else:
    knowledge_triples = []
    print("未找到知识池文件，将使用空知识池。")

已加载思维导图。
已加载知识池，共 168 条三元组。


In [3]:
# 单元格 3：辅助函数 —— 后序遍历树节点
def postorder_traverse(nodes):
    for node in nodes:
        if node.get("children"):
            yield from postorder_traverse(node["children"])
        yield node

In [4]:
# 单元格 4：格式化知识三元组为文本（与原项目类似，但三元组格式不同）
def format_knowledge(triples, max_len=50):
    lines = []
    for s, r, o in triples[:max_len]:
        lines.append(f"({s}, {r}, {o})")  # 原项目格式为 (entity1, rel, entity2)
    return "\n".join(lines)

In [5]:
# 单元格 5：第一次推理函数（基于原项目 reasoning_head + prompt_reason）
def first_reasoning(question, knowledge_triples, verified_answers):
    kg_text = format_knowledge(knowledge_triples)
    verified_text = json.dumps(verified_answers, ensure_ascii=False, indent=2)
    
    prompt = fill_prompt(
        REASONING_PROMPT,
        question=question,
        knowledge_triples=kg_text,
        verified_answers=verified_text
    )
    
    response = ollama_chat(prompt, temperature=0.1, format_json=False)
    # 提取 JSON 对象
    try:
        # 尝试直接解析整个响应
        data = json.loads(response)
        return data.get("answer", "解析失败")
    except:
        # 正则提取
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                data = json.loads(match.group(0))
                return data.get("answer", "解析失败")
            except:
                pass
        return "解析失败"

In [6]:
# 单元格 6：第二次推理函数（用于一致性检查，与原项目相同，但可以用稍高温度）
def second_reasoning(question, knowledge_triples, verified_answers):
    kg_text = format_knowledge(knowledge_triples)
    verified_text = json.dumps(verified_answers, ensure_ascii=False, indent=2)
    
    prompt = fill_prompt(
        SECOND_REASONING_PROMPT,
        question=question,
        knowledge_triples=kg_text,
        verified_answers=verified_text
    )
    
    response = ollama_chat(prompt, temperature=0.2, format_json=False)
    try:
        data = json.loads(response)
        return data.get("answer", "解析失败")
    except:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                data = json.loads(match.group(0))
                return data.get("answer", "解析失败")
            except:
                pass
        return "解析失败"

In [7]:
# 单元格 7：重新思考函数（Rethink）
def rethink(question, knowledge_triples, verified_answers):
    kg_text = format_knowledge(knowledge_triples)
    verified_text = json.dumps(verified_answers, ensure_ascii=False, indent=2)
    
    prompt = fill_prompt(
        RETHINK_PROMPT,
        question=question,
        knowledge_triples=kg_text,
        verified_answers=verified_text
    )
    
    response = ollama_chat(prompt, temperature=0.1, format_json=False)
    try:
        data = json.loads(response)
        return data.get("answer", "解析失败")
    except:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                data = json.loads(match.group(0))
                return data.get("answer", "解析失败")
            except:
                pass
        return "解析失败"

In [8]:
# 单元格 8：综合回答生成函数（用于根问题）
def synthesize_final_answer(root_question, child_answers, knowledge_triples):
    """
    基于子问题答案和知识三元组，生成一个完整、连贯的最终答案。
    """
    kg_text = format_knowledge(knowledge_triples)
    answers_text = ""
    for q, ans in child_answers.items():
        answers_text += f"子问题：{q}\n答案：{ans}\n\n"
    
    prompt = f"""你是一个专业的知识整合与问答助手。你的任务是根据以下提供的子问题答案和相关知识三元组，对原始复杂问题生成一个完整、准确、自然的回答。

**回答要求：**
1. **综合所有子问题答案**：回答必须覆盖原始问题中的所有子问题要点，不能遗漏任何一个。
2. **结合知识图谱事实**：如果知识三元组中包含与问题相关的背景信息或细节，可以适当补充，使回答更加丰富、专业。
3. **严格基于给定信息**：禁止编造任何未在子问题答案或知识三元组中出现的内容。如果某些信息不足以支撑完整的叙述，请如实说明或留白，不要臆测。
4. **语言流畅自然**：用一段或几段连贯的文字表述，避免简单罗列或问答形式。使用中文输出。
5. **结构清晰**：可以按照问题逻辑分点陈述，但整体要融入段落中。

原始问题：{root_question}

已解答的子问题及对应答案：
{answers_text}

相关知识三元组（可作为补充背景）：
{kg_text}

请综合上述信息，用一段通顺的文字回答原始问题。答案应涵盖所有子问题的要点，并可适当补充知识三元组中的相关信息。不要简单罗列，要形成一段完整的叙述。

最终答案："""
    
    response = ollama_chat(prompt, temperature=0.3, format_json=False)
    return response.strip()

In [9]:
# 单元格 9：主推理循环
verified_answers = {}  # 存储已确认的答案，key为问题文本，value为答案字符串

print("开始自底向上推理...\n")

for node in postorder_traverse(mind_map_tree):
    q = node["sub_question"]
    state = node["state"]
    children = node.get("children", [])
    
    # 如果是非叶子节点（需要继续分解），则跳过推理，留待综合模块处理
    if state == "Continue" or children:
        print(f"跳过非叶子节点：{q} (将在综合阶段处理)")
        continue
    
    print(f"正在处理：{q}")
    
    # 第一次推理
    ans1 = first_reasoning(q, knowledge_triples, verified_answers)
    print(f"  第一次推理答案：{ans1}")
    
    # 第二次推理
    ans2 = second_reasoning(q, knowledge_triples, verified_answers)
    print(f"  第二次推理答案：{ans2}")
    
    # 比较两次答案是否一致
    if ans1.strip().lower() == ans2.strip().lower():
        final_ans = ans1
        print("  两次答案一致，采纳。")
    else:
        print("  答案不一致，触发重新思考...")
        final_ans = rethink(q, knowledge_triples, verified_answers)
        print(f"  重新思考答案：{final_ans}")
    
    verified_answers[q] = final_ans
    print()

print("===== 子问题推理完成 =====\n")

开始自底向上推理...

正在处理：陈氏太极拳的动作要素有哪些？
  第一次推理答案：信息不足，无法回答
  第二次推理答案：信息不足，无法回答
  两次答案一致，采纳。

正在处理：陈氏太极拳的代表传承人有哪些？
  第一次推理答案：陈有本、王西安、陈小星、陈照奎、陈延熙、陈正雷、陈照丕、陈小旺、陈庆州、陈王廷
  第二次推理答案：陈有本、王西安、陈小星、陈照奎、陈延熙、陈庆州、陈照丕、陈小旺、陈正雷
  答案不一致，触发重新思考...
  重新思考答案：陈有本、王西安、陈小星、陈照奎、陈延熙、陈庆州、陈正雷、陈照丕、陈小旺

跳过非叶子节点：陈氏太极拳的动作要素有哪些？陈氏太极拳的代表传承人有哪些？ (将在综合阶段处理)
===== 子问题推理完成 =====



In [10]:
# 单元格 10：处理根问题（综合生成最终答案）
# 假设思维导图的第一个节点为根问题
root_node = mind_map_tree[0]
root_question = root_node["sub_question"]

# 收集所有直接子问题的答案（如果是复杂树，可递归收集，这里简单起见取 verified_answers 中非根问题的所有答案）
child_answers = {q: ans for q, ans in verified_answers.items() if q != root_question}

print("正在综合生成最终答案...")
final_answer = synthesize_final_answer(root_question, child_answers, knowledge_triples)
verified_answers[root_question] = final_answer

print("\n===== 最终答案 =====")
print(final_answer)

正在综合生成最终答案...

===== 最终答案 =====
关于陈氏太极拳的代表传承人，现有资料明确列出了陈有本、王西安、陈小星、陈照奎、陈延熙、陈庆州、陈正雷、陈照丕以及陈小旺等多位关键人物，他们均在陈氏太极拳的传承体系中占据重要地位。太极拳作为一种蕴含深厚文化底蕴的体育项目，展现了身心健康、太极文化、养生观以及天人合一等理念，并作为非物质文化遗产得到广泛传承，其发展格局中融合了儒家思想与太极理论，参与了多项国际交流活动。不过，针对陈氏太极拳具体的动作要素有哪些这一问题，基于当前提供的信息资料，并未包含相关的具体动作描述，因此无法对此进行具体说明。


In [11]:
# 单元格 11：输出详细推理链
print("\n详细推理链：")
for q, ans in verified_answers.items():
    print(f"Q: {q}")
    print(f"A: {ans}\n")


详细推理链：
Q: 陈氏太极拳的动作要素有哪些？
A: 信息不足，无法回答

Q: 陈氏太极拳的代表传承人有哪些？
A: 陈有本、王西安、陈小星、陈照奎、陈延熙、陈庆州、陈正雷、陈照丕、陈小旺

Q: 陈氏太极拳的动作要素有哪些？陈氏太极拳的代表传承人有哪些？
A: 关于陈氏太极拳的代表传承人，现有资料明确列出了陈有本、王西安、陈小星、陈照奎、陈延熙、陈庆州、陈正雷、陈照丕以及陈小旺等多位关键人物，他们均在陈氏太极拳的传承体系中占据重要地位。太极拳作为一种蕴含深厚文化底蕴的体育项目，展现了身心健康、太极文化、养生观以及天人合一等理念，并作为非物质文化遗产得到广泛传承，其发展格局中融合了儒家思想与太极理论，参与了多项国际交流活动。不过，针对陈氏太极拳具体的动作要素有哪些这一问题，基于当前提供的信息资料，并未包含相关的具体动作描述，因此无法对此进行具体说明。



In [12]:
# 单元格 12：保存结果
output_file = "data/reasoning_result.json"
result_data = {
    "root_question": root_question,
    "final_answer": final_answer,
    "verified_answers": verified_answers
}
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(result_data, f, ensure_ascii=False, indent=2)
print(f"推理结果已保存至：{output_file}")

推理结果已保存至：data/reasoning_result.json
